In [1]:
# codigo inicial que unificou as bases de vendas e vistoria, criou as colunas de ágio absoluto e percentual, 
# e removeu os registros duplicados dos itens agrupados
# removeu os imoveis da tipologia apartamento/nao é caracteristica da empresa. Venda por convenio de outra instituicao
# teste usando o codigo da destinacao original, via onehot encoding

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error, r2_score, mean_absolute_percentage_error,
    root_mean_squared_error, mean_squared_error
)

import matplotlib.pyplot as plt

tabela = pd.read_csv("base_vendas_atividade2_final1.csv", sep=";", encoding="latin-1")
vistoria = pd.read_csv("base_vistorias_atividade2_final.csv", sep=";", encoding="latin-1")
destinacoes = pd.read_csv("base_destinacoes_atividade2_final.csv", sep=";", encoding="latin-1")

tabela = tabela.merge(
    vistoria[["CD_IMOVEL_URBANO", "SIM_SITIMO_DS"]],
    how="left",
    left_on="CD_IMOVEL",
    right_on="CD_IMOVEL_URBANO"
)


tabela = tabela.merge(
    destinacoes[["COD_DESTINACAO_IMOVEL", "RESIDENCIAL", "COMERCIAL", "INDUSTRIAL", "INSTITUCIONAL"]],
    how="left",
    left_on="COD_DESTINACAO_IMOVEL",
    right_on="COD_DESTINACAO_IMOVEL"
)       


# # remove a coluna duplicada da chave
tabela = tabela.drop(columns=["CD_IMOVEL_URBANO", "CD_IMOVEL"])
tabela = tabela.rename(columns={"SIM_SITIMO_DS": "SITUACAO_VISTORIA"})
# tabela["SITUACAO_VISTORIA"] = tabela["SITUACAO_VISTORIA"].fillna("SEM_VISTORIA")

condicao = (tabela["ANO_VENDA"] >= 2022)
tabela = tabela[condicao]

# # colunas que definem a duplicidade: mesmo ano, mesmo edital e mesmo item do edital (item de edital com imoveis agrupados)
cols = ["ANO_VENDA", "NR_EDITAL", "ITEM_EDITAL"]
# # máscara: True para linhas que aparecem em duplicidade (em qualquer posição do grupo)
mask_dup = tabela.duplicated(subset=cols, keep=False)
# # remove TODAS as linhas duplicadas desses grupos
tabela_sem_dups = tabela.loc[~mask_dup].copy()

# # Foram removidos 145 itens que constavam como items agrupados, de um total de 2982, restando 2837 registros 
print("Linhas originais:", len(tabela))
print("Linhas removidas:", mask_dup.sum())
print("Linhas finais:", len(tabela_sem_dups))

tabela = tabela_sem_dups
tabela["NR_EDITAL"] = tabela["ANO_VENDA"].astype(str) + "-" + tabela["NR_EDITAL"].astype(str)

def br_to_float(s):
    """
    Converte número no formato BR para float:
    - remove separador de milhar (.)
    - troca decimal (,) por (.)
    """
    if pd.isna(s):
        return np.nan
    s = str(s).strip()
    if s == "":
        return np.nan
    s = s.replace(".", "")      # remove milhares
    s = s.replace(",", ".")     # troca decimal
    return pd.to_numeric(s, errors="coerce")

cols = ["VALOR_VENDA", "AREA_MAX_CONSTR", "AREA_BASE", "AREA", "VALOR_LAUDO"]  # ajuste
for c in cols:
    if c in tabela.columns:
        tabela[c] = tabela[c].apply(br_to_float)

# #criacao da coluna AGIO ABSOLUTO , esta coluna deve ser retirada do treino
tabela["AGIO_ABSOLUTO"] = tabela["VALOR_VENDA"] - tabela["VALOR_LAUDO"]
tabela["AGIO_PERCENTUAL"] = ((tabela["VALOR_VENDA"] - tabela["VALOR_LAUDO"]) / tabela["VALOR_LAUDO"]) * 100

#retirando os apartamentos da base (14 registros) que tem área máxima de construção igual a zero e área base igual a zero, ou seja, não tem área construída, o que é um erro de cadastro
tabela = tabela[~((tabela["AREA_MAX_CONSTR"] == 0) & (tabela["AREA_BASE"] == 0))]   
# Percentual 0–100 -> 0–1
tabela["PERCENTUAL_ENTRADA"] = pd.to_numeric(tabela["PERCENTUAL_ENTRADA"], errors="coerce") / 100.0
condicao = (tabela["ANO_VENDA"] >= 2022)
tabela = tabela[condicao]
print(f"Maior Valor de Venda: {tabela['VALOR_VENDA'].max():.2f}")
print(f"Menor Valor de Venda: {tabela['VALOR_VENDA'].min():.2f}")
print(f"Média do Valor da Venda: {tabela['VALOR_VENDA'].mean():.2f}")
print(f"Desvio Padrão do Valor da Venda: {tabela['VALOR_VENDA'].std():.2f}")






Linhas originais: 2076
Linhas removidas: 118
Linhas finais: 1958
Maior Valor de Venda: 41451000.00
Menor Valor de Venda: 23000.01
Média do Valor da Venda: 815405.20
Desvio Padrão do Valor da Venda: 2266013.92


In [2]:
# -------------------------
# 0) DADOS (ajuste se necessário)
# -------------------------
X = tabela.drop(columns=["DS_CIDADE", "AREA_MAX_CONSTR", "VALOR_VENDA", "AGIO_ABSOLUTO", "AGIO_PERCENTUAL", "VALOR_LAUDO", "AREA_BASE", "QTD_OFERTAS", "COD_DESTINACAO_IMOVEL", "ANO_VENDA", "NR_EDITAL", "ITEM_EDITAL"])
y = tabela["VALOR_VENDA"]

# Definir colunas numéricas e categóricas explicitamente
colunas_numericas = ["AREA", "PERCENTUAL_ENTRADA", "RESIDENCIAL", "COMERCIAL", "INDUSTRIAL", "INSTITUCIONAL"]
colunas_categoricas = ["DS_SETOR", "SITUACAO_VISTORIA"]


print("Numéricas:", colunas_numericas)
print("Categóricas:", colunas_categoricas)
print("Shape X:", X.shape, "| Shape y:", y.shape)

Numéricas: ['AREA', 'PERCENTUAL_ENTRADA', 'RESIDENCIAL', 'COMERCIAL', 'INDUSTRIAL', 'INSTITUCIONAL']
Categóricas: ['DS_SETOR', 'SITUACAO_VISTORIA']
Shape X: (1948, 8) | Shape y: (1948,)


In [4]:
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("numero de linhas e colunas dos dados de treino:", X_treino.shape)
print("numero de linhas e colunas dos dados de treino:", X_teste.shape)


# etapas_numericas

escalonador = StandardScaler()
imputador_numerico = SimpleImputer(strategy="median")

etapas_numericas = Pipeline(
    [
        ("imputer", imputador_numerico),
        ("scaler", escalonador)
    ]
)

#etapas categoricas

categorizador = OneHotEncoder(handle_unknown="ignore")
imputador_categorico = SimpleImputer(strategy="most_frequent")

etapas_categoricas = Pipeline(
    [
        ("imputer", imputador_categorico),
        ("encoder", categorizador)
    ]
)


preprocessador = ColumnTransformer(
    transformers=[
        ('numericas', etapas_numericas, colunas_numericas),
        ('categoricas', etapas_categoricas, colunas_categoricas)
    ]
)

x_treino_transformado = preprocessador.fit_transform(X_treino)
x_teste_transformado = preprocessador.transform(X_teste)


modelo = RandomForestRegressor(random_state=42, n_jobs=-1)

# param_grid = {
#     "n_estimators": [300, 600, 1000],       # quantidade de árvores
#     "max_depth": [None, 10, 20, 30],        # profundidade máxima
#     "min_samples_split": [2, 5, 10, 20],    # mínimo para dividir nó
#     "min_samples_leaf": [1, 2, 5, 10],      # mínimo por folha (ajuda a reduzir overfit)
#     "max_features": ["sqrt", 0.5, 0.7, 1.0] # qtd de features por split
# }

param_grid = {
    "n_estimators": [300],       # quantidade de árvores
    "max_depth": [30],        # profundidade máxima
    "min_samples_split": [2],    # mínimo para dividir nó
    "min_samples_leaf": [1],      # mínimo por folha (ajuda a reduzir overfit)
    "max_features": [0.7] # qtd de features por split
}


grid_search = GridSearchCV(
    estimator=modelo,
    param_grid=param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    verbose=1
)

grid_search.fit(x_treino_transformado, y_treino)
melhor_modelo = grid_search.best_estimator_

# --- TREINO
y_pred_treino_rf = melhor_modelo.predict(x_treino_transformado)

rmse_treino_rf = root_mean_squared_error(y_treino, y_pred_treino_rf)
mae_treino_rf = mean_absolute_error(y_treino, y_pred_treino_rf)
r2_treino_rf = r2_score(y_treino, y_pred_treino_rf)
mape_treino_rf = mean_absolute_percentage_error(y_treino, y_pred_treino_rf)

print("Melhores parametros encontrados:", grid_search.best_params_)
print("R2 no treino: ", r2_treino_rf)
print("RMSE no treino: ", rmse_treino_rf)
print("MAE no treino: ", mae_treino_rf)
print("MAPE no treino (%): ", mape_treino_rf)

# --- TESTE
y_pred_teste_rf = melhor_modelo.predict(x_teste_transformado)

rmse_teste_rf = root_mean_squared_error(y_teste, y_pred_teste_rf)
mae_teste_rf = mean_absolute_error(y_teste, y_pred_teste_rf)
r2_teste_rf = r2_score(y_teste, y_pred_teste_rf)
mape_teste_rf = mean_absolute_percentage_error(y_teste, y_pred_teste_rf)

print("\n-------------------------")
print("Avaliacao do modelo RandomForest no teste:")
print("R2 no teste: ", r2_teste_rf)
print("RMSE no teste: ", rmse_teste_rf)
print("MAE no teste: ", mae_teste_rf)
print("MAPE no teste (%): ", mape_teste_rf)

numero de linhas e colunas dos dados de treino: (1558, 8)
numero de linhas e colunas dos dados de treino: (390, 8)
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Melhores parametros encontrados: {'max_depth': 30, 'max_features': 0.7, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}
R2 no treino:  0.9539981684101602
RMSE no treino:  498807.7993819036
MAE no treino:  112570.07659539364
MAPE no treino (%):  0.1400961968717498

-------------------------
Avaliacao do modelo RandomForest no teste:
R2 no teste:  0.8945407334144416
RMSE no teste:  651238.423122678
MAE no teste:  180424.55477267466
MAPE no teste (%):  0.2151377156711475
